# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I used a line chart because it makes it easy to compare population growth across continents over time. I used 1970 as the starting point (100) to compare the relative growth of each continent.

In [2]:

continent_pop = df_long.groupby(['Continent', 'Year'])['Population'].sum().reset_index()

base = continent_pop[continent_pop['Year'] == 1970][['Continent', 'Population']]
base = base.rename(columns={'Population': 'Base'})

continent_pop = continent_pop.merge(base, on='Continent')
continent_pop['Growth'] = continent_pop['Population'] / continent_pop['Base'] * 100

fig = px.line(continent_pop, x='Year', y='Growth', color='Continent',
              title='Africa Has the Fastest Population Growth Since 1970',
              template='plotly_white')

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I used a sorted bar chart because it makes it easy to compare the 15 countries with the highest population increase. I used different colours to highlight the African countries.

In [4]:

df['Increase'] = df['2022 Population'] - df['1970 Population']

top15 = df.nlargest(15, 'Increase')

fig = px.bar(top15.sort_values('Increase'),
             x='Increase',
             y='Country',
             color='Continent',
             orientation='h',
             title='Top 15 Countries by Population Increase',
             template='plotly_white')

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

print("African countries:",
      (top15['Continent'] == 'Africa').sum())


African countries: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I used a scatter plot because it helps to show the relationship between land area and population. I used different colours for each continent to compare them and identify countries that do not follow the general pattern.

In [5]:

fig = px.scatter(df, x='Area', y='2022 Population',
                 color='Continent',
                 hover_name='Country',
                 log_x=True, log_y=True,
                 title='Land Area and Population by Continent',
                 template='plotly_white')

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I used a box plot because it shows the distribution and spread of population density across different continents. It also helps to identify countries with unusually high or low population density.

In [6]:

fig = px.box(df, x='Continent', y='Density',
             color='Continent',
             title='Population Density Across Continents',
             template='plotly_white')

fig.update_layout(title_x=0.5, showlegend=False)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I used a choropleth map because it shows population changes geographically across countries. The colour scale makes it easy to identify countries where the population increased or decreased.

In [12]:


df['Change (%)'] = ((df['2022 Population'] - df['2000 Population']) / df['2000 Population']) * 100

fig = px.choropleth(df, locations='CCA3',
                    locationmode='ISO-3',
                    color='Change (%)',
                    hover_name='Country',
                    color_continuous_scale='RdBu',
                    color_continuous_midpoint=0,
                    title='Population Changes Across the World',
                    template='plotly_white')

fig.update_layout(title_x=0.5)
fig.show()



---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I used a stacked bar chart to compare the population distribution across continents and countries. Different colours represent the countries, making it easier to compare their contribution to the total population.

In [10]:

fig = px.bar(df, x='Continent', y='2022 Population',
             color='Country',
             title='World Population Distribution by Continent',
             template='plotly_white')

fig.update_layout(title_x=0.5, barmode='stack')
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()



---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I used a line chart because it clearly shows how the population share of each continent changes over time. It makes it easy to compare the trends and identify which continents are gaining or losing their share of the world's population.

In [11]:

continent_pop = df_long.groupby(['Year', 'Continent'])['Population'].sum().reset_index()

continent_pop['Continent'] = continent_pop['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

continent_pop = continent_pop.groupby(['Year', 'Continent'])['Population'].sum().reset_index()

continent_pop['Share'] = continent_pop.groupby('Year')['Population'].transform(
    lambda x: x / x.sum() * 100
)

fig = px.line(continent_pop, x='Year', y='Share',
              color='Continent',
              title='Changes in World Population Share (1970-2022)',
              template='plotly_white')

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, title='Population Share (%)')

fig.show()
